# 03 · Phân lớp (Classification) + Cross-Validation + F-score

## Mục tiêu
Phân lớp nhị phân `status` (0 = khỏe, 1 = Parkinson) từ 22 feature giọng nói. Đánh giá bằng 10-fold Cross-Validation và F-score, so sánh đúng 3 thuật toán: KNN, SVM, Random Forest.

## Phạm vi
- M05: thiết lập CV (M05-T1), chạy 3 mô hình và thu F-score (M05-T2), báo cáo ở `results/classification_report.md` (M05-T3).
- Không tự kết luận mô hình tốt nhất, để reviewer và M06 cùng kết luận.
- Reviewer: M05-T1 → M01 · M05-T2 → M01 (gate) + M06 · M05-T3 → M06.

## Điều kiện bắt đầu
- Đầu vào: canonical CSV `data/processed/parkinsons_v1.csv` (22 feature + `status`, đã bỏ `name`).
- Chạy từ gốc repo hoặc từ bất kỳ thư mục con nào của repo; notebook tự tìm gốc project.

## Hai điểm đã chốt
- Số fold: k = 10 (StratifiedKFold, seed 42) theo đề bài; `docs/REPRODUCIBILITY.md` đang ghi k = 5, M01 cập nhật.
- Loại F-score: macro là chỉ số chính (dữ liệu lệch lớp 147/48), weighted ghi kèm để đối chiếu.

In [ ]:
import datetime
import json
from pathlib import Path

import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Chạy riêng trên máy: đặt CSV_OVERRIDE = r"D:\parkinsons_v1.csv". Khi nộp lên git: để None.
CSV_OVERRIDE = None

if CSV_OVERRIDE:
    CSV = Path(CSV_OVERRIDE)
    OUT_DIR = CSV.parent / "results"
else:
    # Tìm gốc project: đi ngược từ thư mục hiện tại đến nơi có canonical CSV
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "data/processed/parkinsons_v1.csv").exists()), None)
    assert ROOT is not None, "Thiếu canonical CSV (S-T1) -> Blocked"
    CSV = ROOT / "data/processed/parkinsons_v1.csv"
    OUT_DIR = ROOT / "results"

SEED = 42
K_OUTER = 10   # số fold (theo đề bài)
K_INNER = 5    # CV bên trong để chọn tham số
print("CSV:", CSV)

## Bước 1 · Biến mục tiêu
Chốt `y = status` (nhị phân 0/1). Kỳ vọng: 195 dòng × 23 cột, lớp 1 có 147 mẫu, lớp 0 có 48 mẫu (lệch lớp).

In [ ]:
df = pd.read_csv(CSV)
y = df["status"]
print(df.shape)
print(y.value_counts())

## Bước 2 · Chuẩn bị X và y
`X` gồm 22 feature, không chứa `status` hay `name`. Dùng `X` gốc, chưa chuẩn hoá: scaler nằm trong `Pipeline` nên chỉ được fit trên fold train.

In [ ]:
X = df.drop(columns=["status"])
assert X.shape[1] == 22 and "name" not in X.columns
assert X.isna().sum().sum() == 0

## Bước 3-4 · Quy trình chống leakage và 10-fold CV
- Scaler fit trên fold train, transform trên fold validation (nằm trong `Pipeline`).
- Chọn tham số bằng `GridSearchCV` chạy bên trong fold train (CV trong, 5-fold), không chạm fold test.
- CV ngoài: `StratifiedKFold(n_splits=10, shuffle=True, random_state=42)`, giữ tỉ lệ 0/1 mỗi fold.

In [ ]:
outer_cv = StratifiedKFold(n_splits=K_OUTER, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=K_INNER, shuffle=True, random_state=SEED)

## Bước 5-6 · Chọn đúng 3/5 mô hình và phạm vi tinh chỉnh
- KNN: baseline dựa trên khoảng cách, thử `n_neighbors` ∈ {3, 5, 7}.
- SVM: ranh giới phân tách có lề lớn, thử `C` ∈ {0.1, 1, 10}.
- Random Forest: tổ hợp nhiều cây quyết định, ít nhạy với thang đo, thử `n_estimators` ∈ {50, 100}.
- Mỗi mô hình chỉ thử một tham số chính, không chạy lưới lớn.

In [ ]:
models = {
    "KNN": (
        Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
        {"clf__n_neighbors": [3, 5, 7]},
    ),
    "SVM": (
        Pipeline([("scaler", StandardScaler()), ("clf", SVC(random_state=SEED))]),
        {"clf__C": [0.1, 1, 10]},
    ),
    "RandomForest": (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", RandomForestClassifier(random_state=SEED))]),
        {"clf__n_estimators": [50, 100]},
    ),
}

## Bước 7 · Chạy CV và thu F-score
F-score macro (chính) và weighted (đối chiếu), mean ± std trên 10 fold. Có thêm điểm train để soi overfit, và tham số tối ưu của từng fold để tái lập đầy đủ.

In [ ]:
scoring = {"f1_macro": "f1_macro", "f1_weighted": "f1_weighted"}
rows = []
for name, (pipe, grid) in models.items():
    search = GridSearchCV(pipe, grid, cv=inner_cv, scoring="f1_macro", n_jobs=-1)
    res = cross_validate(search, X, y, cv=outer_cv, scoring=scoring,
                         return_train_score=True, return_estimator=True)
    best = [est.best_params_ for est in res["estimator"]]
    rows.append({
        "model": name,
        "f1_macro_test": f"{res['test_f1_macro'].mean():.4f} ± {res['test_f1_macro'].std():.4f}",
        "f1_weighted_test": f"{res['test_f1_weighted'].mean():.4f} ± {res['test_f1_weighted'].std():.4f}",
        "f1_macro_train": f"{res['train_f1_macro'].mean():.4f}",
        "grid": str(grid),
        "best_params_per_fold": json.dumps(best),
    })
table = pd.DataFrame(rows)
print(table.drop(columns="best_params_per_fold").to_string(index=False))

## Lưu kết quả vào `results/`
Ghi kèm tham số (k, seed, ngày chạy, phiên bản thư viện) theo `HANDOFF.md`.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
table["date"] = datetime.date.today().isoformat()
table["k_outer"], table["k_inner"], table["seed"] = K_OUTER, K_INNER, SEED
table["sklearn"] = sklearn.__version__
table.to_csv(OUT_DIR / "m05_f1_cv.csv", index=False)
print("Đã lưu:", OUT_DIR / "m05_f1_cv.csv")

## Kết quả và nhận xét
Số liệu dưới đây lấy từ lần chạy trên canonical v1 (seed 42, k ngoài 10, k trong 5); chi tiết ở `results/m05_f1_cv.csv`.

| Mô hình | F-score macro (test) | F-score weighted (test) | F-score macro (train) |
|---|:-:|:-:|:-:|
| KNN | 0.8757 ± 0.0742 | 0.9073 ± 0.0578 | 0.9647 |
| SVM | 0.8904 ± 0.0804 | 0.9199 ± 0.0603 | 0.9787 |
| Random Forest | 0.8737 ± 0.1171 | 0.9103 ± 0.0824 | 1.0000 |

- Chênh lệch giữa 3 mô hình (khoảng 0.016) nhỏ hơn độ lệch chuẩn giữa các fold (0.07–0.12).
- F-score weighted cao hơn macro ở cả 3 mô hình, phù hợp với việc lớp khỏe chỉ có 48 mẫu.
- Điểm train cao hơn test ở cả 3 mô hình; khoảng cách lớn nhất ở Random Forest (train 1.0000, test 0.8737).
- Kết luận mô hình nào tốt nhất: để reviewer và M06 quyết, không ghi ở đây.

## Hạn chế
- Dữ liệu nhỏ (195 mẫu), mỗi fold test chỉ khoảng 20 mẫu, trong đó khoảng 5 mẫu lớp khỏe.
- Lệch lớp 147/48.
- Dataset có nhiều bản ghi của cùng một người (khoảng 31 người), cột `name` đã bị loại nên `StratifiedKFold` có thể đặt bản ghi của cùng một người vào cả train và test; điểm test có thể cao hơn thực tế.

## Checklist
- [x] `y = status`, `X` = 22 feature, không có `name`/`status`.
- [x] Pipeline chống leakage; chọn tham số chỉ trên fold train; không dùng test để chọn tham số.
- [x] 10-fold stratified CV (seed 42, k = 10).
- [x] Đúng 3/5 mô hình; F-score mean ± std từ kết quả thật.
- [x] Kết quả lưu ở `results/m05_f1_cv.csv`, báo cáo ở `results/classification_report.md`.

## Tái lập
Seed 42 · k ngoài = 10 · k trong = 5 · canonical CSV v1 · phiên bản thư viện: xem cột `sklearn` trong `results/m05_f1_cv.csv`.